[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/43.%20The%20Security%20Inspection%20Optimization%20Problem/P43-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 43. The Security Inspection Optimization Problem

## Tier 4: Neural Architecture Search with an Evolutionary Algorithm

### Goal

This notebook demonstrates Neural Architecture Search (NAS): an evolutionary algorithm searches a space of candidate neural-network architectures and scores each candidate using a simulated accuracy estimate and efficiency measures. The aim is to understand how architecture representations, fitness evaluation, selection, crossover, mutation, and elitism work together in a search process.

### Inputs, outputs, and scope

The inputs are a fixed image shape, a number of output classes, and an evolutionary-search configuration. The search returns a candidate architecture, its simulated accuracy, parameter count, inference-time estimate, and fitness score, together with histories used for analysis and visualization.

**This is an algorithmic demonstration, not a trained threat-detection system.** The notebook does not load X-ray images, train a neural network, or measure accuracy on a validation or test set. Its sample counts, class characteristics, baseline, accuracy, and inference time are illustrative inputs to a proxy model. Consequently, reported search outcomes are not empirical evidence of detection performance.

### Model assumptions

- Candidate networks are sequential chains of convolution, pooling, attention, dropout, and dense layers. A convolutional feature-extraction layer precedes the classifier; spatial layers occur before any dense layer.
- Convolutions use valid-padding shape calculations. Pooling uses integer division. Dense layers flatten spatial features when needed. The attention parameter count is a simplified projection estimate, not a complete specification of an attention implementation.
- Accuracy is generated by a hand-designed proxy with random variation. Inference time is estimated from parameter count and layer count. Neither quantity is measured on hardware or data.
- The evolutionary search uses tournament selection, crossover, mutation, and elitism. Because fitness evaluations contain random variation and no random seed is specified, repeated runs need not produce the same result.
- The search-space configuration includes maximum filter and dense-unit settings, while the candidate choices are explicitly listed in the search class. The illustrative search does not train architectures or guarantee a globally optimal architecture.

### Search method

An **architecture** is an ordered list of layer configurations. A **population** is a collection of candidate architectures. In each generation, the search retains the highest-fitness candidates (elitism), selects parents by small random tournaments, optionally recombines their layer sequences (crossover), and applies random structural changes (mutation). Fitness determines preference during selection; it is not itself a probability of correct threat detection.

Let $P_g$ denote the population in generation $g$, and let $F(a)$ be the fitness of architecture $a$. The best candidate in a population is

$$a_g^* = \operatorname*{arg\,max}_{a \in P_g} F(a).$$

The fitness combines the proxy accuracy $A(a)$ and efficiency terms:

$$F(a) = 0.7 A(a) + 0.3 E(a) Q(a),$$

where $E(a)=\min(1, 0.1/t(a))$ rewards lower estimated inference time $t(a)$, and $Q(a)$ penalizes parameter counts above the stated threshold. These are dimensionless scores; the weights and thresholds are demonstration choices, not learned quantities. In the code, `calculate_fitness` implements this expression, `evolve_population` constructs the next population, and `search` records generation histories and returns the best candidate encountered.

If the population size is $N$ and there are $G$ generations, each generation evaluates offspring as well as the initial population. With $e$ elites retained without reevaluation, the search performs $N + G(N-e)$ fitness evaluations. The cost of an evaluation depends on the candidate architecture; here it is a lightweight simulation rather than neural-network training. The number of possible layer sequences grows rapidly with sequence length and the number of layer configurations, which motivates guided search rather than exhaustive enumeration.

### How to interpret the results

The best architecture is the highest-fitness candidate encountered by this stochastic search, not a proven global optimum. Accuracy and inference time are outputs of the notebook's proxy formulas. Diversity is a simple structural summary based on layer counts and layer-type sets; it is not a measure of functional or predictive diversity. The baseline is a reference value used by the simulation, not a result from an experiment in this notebook.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Optional, Tuple
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class LayerConfig:
    """Configuration for one layer in a sequential candidate network."""

    layer_type: str
    filters: int = 0
    units: int = 0
    kernel_size: int = 3
    stride: int = 1
    activation: str = "relu"
    dropout_rate: float = 0.0
    pool_size: int = 2


@dataclass
class NeuralArchitecture:
    """A candidate network and its simulated evaluation metrics."""

    layers: List[LayerConfig]
    input_shape: Tuple[int, int, int] = (224, 224, 3)
    num_classes: int = 8
    fitness: float = 0.0
    accuracy: float = 0.0
    parameters: int = 0
    inference_time: float = 0.0

    def calculate_parameters(self) -> int:
        """Count trainable parameters using the notebook's layer assumptions."""
        total_params = 0
        current_shape = self.input_shape

        for layer in self.layers:
            if layer.layer_type == "conv":
                if not isinstance(current_shape, tuple) or len(current_shape) != 3:
                    raise ValueError("A convolutional layer requires a spatial input.")
                height, width, input_channels = current_shape
                output_h = (height - layer.kernel_size) // layer.stride + 1
                output_w = (width - layer.kernel_size) // layer.stride + 1
                if output_h < 1 or output_w < 1:
                    raise ValueError("The convolution kernel does not fit the input shape.")
                total_params += (
                    layer.filters * layer.kernel_size**2 * input_channels
                    + layer.filters
                )
                current_shape = (output_h, output_w, layer.filters)

            elif layer.layer_type == "dense":
                input_size = int(np.prod(current_shape))
                total_params += input_size * layer.units + layer.units
                current_shape = (layer.units,)

            elif layer.layer_type == "pool":
                if not isinstance(current_shape, tuple) or len(current_shape) != 3:
                    raise ValueError("A pooling layer requires a spatial input.")
                height, width, channels = current_shape
                output_h = height // layer.pool_size
                output_w = width // layer.pool_size
                if output_h < 1 or output_w < 1:
                    raise ValueError("The pooling window does not fit the input shape.")
                current_shape = (output_h, output_w, channels)

            elif layer.layer_type == "attention":
                if not isinstance(current_shape, tuple) or len(current_shape) != 3:
                    raise ValueError("The simplified attention layer requires a spatial input.")
                channels = current_shape[-1]
                total_params += 4 * channels * channels

            elif layer.layer_type != "dropout":
                raise ValueError(f"Unsupported layer type: {layer.layer_type}")

        return total_params


@dataclass
class NASConfig:
    """Settings for the evolutionary architecture search."""

    population_size: int = 20
    generations: int = 10
    mutation_rate: float = 0.3
    crossover_rate: float = 0.7
    elite_size: int = 4
    max_layers: int = 10
    max_filters: int = 512
    max_dense_units: int = 1024


class NeuralArchitectureSearch:
    """Evolutionary NAS demonstration for simulated security-image scoring."""

    def __init__(
        self,
        config: NASConfig,
        input_shape: Tuple[int, int, int],
        num_classes: int,
    ):
        if config.population_size < 1:
            raise ValueError("population_size must be at least 1.")
        if not 0 <= config.elite_size <= config.population_size:
            raise ValueError("elite_size must be between 0 and population_size.")
        if config.generations < 0 or config.max_layers < 3:
            raise ValueError("generations must be nonnegative and max_layers at least 3.")
        if len(input_shape) != 3 or min(input_shape) < 1 or num_classes < 2:
            raise ValueError("Use a positive 3D input shape and at least two classes.")

        self.config = config
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.conv_filters = [32, 64, 128, 256, 512]
        self.kernel_sizes = [3, 5, 7]
        self.activations = ["relu", "gelu", "swish"]
        self.dense_units = [64, 128, 256, 512, 1024]
        self.dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.5]

        self.best_accuracy_history = []
        self.avg_accuracy_history = []
        self.diversity_history = []
        self.best_architecture = None
        self.setup_simulated_data()

    def setup_simulated_data(self):
        """Store illustrative data characteristics for the proxy evaluation."""
        self.num_train_samples = 50000
        self.num_val_samples = 10000
        self.num_test_samples = 10000
        self.class_distribution = [0.15, 0.10, 0.08, 0.12, 0.20, 0.25, 0.07, 0.03]
        self.class_difficulty = [0.85, 0.92, 0.78, 0.88, 0.75, 0.70, 0.95, 0.98]
        self.baseline_accuracy = 0.932
        self.baseline_params = 22000000

    def generate_random_layer(self) -> LayerConfig:
        """Sample a layer configuration from the demonstration search space."""
        layer_type = random.choice(
            ["conv", "conv", "conv", "pool", "attention", "dropout"]
        )
        if layer_type == "conv":
            return LayerConfig(
                layer_type="conv",
                filters=random.choice(self.conv_filters),
                kernel_size=random.choice(self.kernel_sizes),
                stride=random.choice([1, 2]),
                activation=random.choice(self.activations),
            )
        if layer_type == "pool":
            return LayerConfig(layer_type="pool", pool_size=random.choice([2, 3]))
        if layer_type == "attention":
            return LayerConfig(layer_type="attention")
        if layer_type == "dropout":
            return LayerConfig(
                layer_type="dropout",
                dropout_rate=random.choice(self.dropout_rates),
            )
        return LayerConfig(
            layer_type="dense",
            units=random.choice(self.dense_units),
            activation=random.choice(self.activations),
        )

    def _classification_layer(self) -> LayerConfig:
        return LayerConfig(
            layer_type="dense",
            units=self.num_classes,
            activation="softmax",
        )

    def _normalize_architecture(
        self, layers: List[LayerConfig]
    ) -> List[LayerConfig]:
        """Keep offspring valid under the sequential shape assumptions."""
        candidates = copy.deepcopy(layers[:-1]) if layers else []
        spatial_layers = []
        dense_layers = []
        dense_phase = False
        shape = self.input_shape

        for layer in candidates:
            if layer.layer_type == "dense":
                dense_phase = True
                dense_layers.append(layer)
                continue
            if dense_phase and layer.layer_type in {"conv", "pool", "attention"}:
                continue

            if layer.layer_type == "conv":
                height, width, channels = shape
                output_h = (height - layer.kernel_size) // layer.stride + 1
                output_w = (width - layer.kernel_size) // layer.stride + 1
                if output_h < 1 or output_w < 1:
                    continue
                shape = (output_h, output_w, layer.filters)
            elif layer.layer_type == "pool":
                height, width, channels = shape
                output_h = height // layer.pool_size
                output_w = width // layer.pool_size
                if output_h < 1 or output_w < 1:
                    continue
                shape = (output_h, output_w, channels)
            elif layer.layer_type == "attention":
                pass
            elif layer.layer_type != "dropout":
                continue

            spatial_layers.append(layer)

        if not any(layer.layer_type == "conv" for layer in spatial_layers):
            spatial_layers.insert(
                0,
                LayerConfig(
                    layer_type="conv",
                    filters=64,
                    kernel_size=3,
                    stride=1,
                    activation="relu",
                ),
            )

        available = max(1, self.config.max_layers - 1)
        interior = (spatial_layers + dense_layers)[:available]
        first_dense = next(
            (i for i, layer in enumerate(interior) if layer.layer_type == "dense"),
            len(interior),
        )
        interior = [
            layer
            for i, layer in enumerate(interior)
            if layer.layer_type == "dense" or i < first_dense
        ]
        return interior + [self._classification_layer()]

    def generate_random_architecture(self) -> NeuralArchitecture:
        """Create a random candidate with a convolutional stem and classifier."""
        num_layers = random.randint(3, self.config.max_layers)
        layers = [self.generate_random_layer() for _ in range(num_layers - 1)]
        layers = self._normalize_architecture(layers + [self._classification_layer()])
        return NeuralArchitecture(
            layers=layers,
            input_shape=self.input_shape,
            num_classes=self.num_classes,
        )

    def simulate_training(
        self, architecture: NeuralArchitecture
    ) -> Tuple[float, float]:
        """Return proxy accuracy and inference time; no model is trained."""
        num_conv_layers = sum(
            layer.layer_type == "conv" for layer in architecture.layers
        )
        num_attention_layers = sum(
            layer.layer_type == "attention" for layer in architecture.layers
        )
        total_filters = sum(
            layer.filters
            for layer in architecture.layers
            if layer.layer_type == "conv"
        )

        conv_bonus = min(0.05, num_conv_layers * 0.01)
        attention_bonus = min(0.08, num_attention_layers * 0.03)
        filter_bonus = min(
            0.04, np.log(total_filters + 1) / np.log(1000) * 0.05
        )
        params = architecture.calculate_parameters()
        efficiency_penalty = (
            max(0, params - 30000000) / 100000000 * 0.02
        )
        random_variation = np.random.normal(0, 0.005)
        accuracy = self.baseline_accuracy + conv_bonus + attention_bonus
        accuracy += filter_bonus - efficiency_penalty + random_variation
        accuracy = max(0.5, min(0.99, accuracy))

        base_inference_time = 0.01
        inference_time = base_inference_time * (params / self.baseline_params)
        inference_time *= len(architecture.layers) / 8
        inference_time = max(0.001, inference_time)
        return accuracy, inference_time

    def calculate_fitness(self, architecture: NeuralArchitecture) -> float:
        """Combine simulated accuracy and normalized efficiency into fitness."""
        accuracy, inference_time = self.simulate_training(architecture)
        architecture.accuracy = accuracy
        architecture.inference_time = inference_time
        architecture.parameters = architecture.calculate_parameters()

        accuracy_weight = 0.7
        efficiency_weight = 0.3
        max_acceptable_time = 0.1
        normalized_efficiency = min(1.0, max_acceptable_time / inference_time)
        param_efficiency = min(1.0, 25000000 / architecture.parameters)
        architecture.fitness = (
            accuracy_weight * accuracy
            + efficiency_weight * normalized_efficiency * param_efficiency
        )
        return architecture.fitness

    def initialize_population(self) -> List[NeuralArchitecture]:
        """Build and evaluate the first generation of candidate architectures."""
        population = []
        for _ in range(self.config.population_size):
            architecture = self.generate_random_architecture()
            self.calculate_fitness(architecture)
            population.append(architecture)
        return population

    def crossover(
        self, parent1: NeuralArchitecture, parent2: NeuralArchitecture
    ) -> NeuralArchitecture:
        """Combine layer sequences from two selected parent architectures."""
        max_layers = min(len(parent1.layers), len(parent2.layers))
        if max_layers <= 2:
            child_layers = copy.deepcopy(random.choice([parent1, parent2]).layers)
        else:
            crossover_point = random.randint(1, max_layers - 1)
            child_layers = (
                copy.deepcopy(parent1.layers[:crossover_point])
                + copy.deepcopy(parent2.layers[crossover_point:])
            )
        child_layers = self._normalize_architecture(child_layers)
        return NeuralArchitecture(
            layers=child_layers,
            input_shape=self.input_shape,
            num_classes=self.num_classes,
        )

    def mutate(self, architecture: NeuralArchitecture) -> NeuralArchitecture:
        """Apply random layer edits while preserving a valid classifier structure."""
        mutated_layers = copy.deepcopy(architecture.layers)
        for _ in range(random.randint(1, 3)):
            if random.random() >= self.config.mutation_rate:
                continue
            operation = random.choice(
                ["modify_layer", "add_layer", "remove_layer", "swap_layers"]
            )
            if operation == "modify_layer" and len(mutated_layers) > 1:
                index = random.randrange(len(mutated_layers) - 1)
                mutated_layers[index] = self.generate_random_layer()
            elif operation == "add_layer" and len(mutated_layers) < self.config.max_layers:
                index = random.randrange(len(mutated_layers))
                mutated_layers.insert(index, self.generate_random_layer())
            elif operation == "remove_layer" and len(mutated_layers) > 3:
                index = random.randrange(1, len(mutated_layers) - 1)
                mutated_layers.pop(index)
            elif operation == "swap_layers" and len(mutated_layers) > 3:
                index = random.randrange(1, len(mutated_layers) - 2)
                mutated_layers[index], mutated_layers[index + 1] = (
                    mutated_layers[index + 1],
                    mutated_layers[index],
                )

        mutated_layers = self._normalize_architecture(mutated_layers)
        return NeuralArchitecture(
            layers=mutated_layers,
            input_shape=self.input_shape,
            num_classes=self.num_classes,
        )

    def calculate_diversity(self, population: List[NeuralArchitecture]) -> float:
        """Average pairwise structural difference in layer counts and types."""
        if len(population) <= 1:
            return 0.0
        total_diversity = 0.0
        comparisons = 0
        for i, architecture1 in enumerate(population):
            for architecture2 in population[i + 1 :]:
                layer_diff = abs(len(architecture1.layers) - len(architecture2.layers))
                types1 = {layer.layer_type for layer in architecture1.layers}
                types2 = {layer.layer_type for layer in architecture2.layers}
                type_diff = len(types1.symmetric_difference(types2))
                total_diversity += layer_diff + type_diff * 0.5
                comparisons += 1
        return total_diversity / comparisons

    def evolve_population(
        self, population: List[NeuralArchitecture]
    ) -> List[NeuralArchitecture]:
        """Create a new generation using elitism, tournaments, crossover, and mutation."""
        population = sorted(population, key=lambda item: item.fitness, reverse=True)
        new_population = [
            copy.deepcopy(item) for item in population[: self.config.elite_size]
        ]

        while len(new_population) < self.config.population_size:
            tournament_size = min(3, len(population))
            tournament1 = random.sample(population, tournament_size)
            tournament2 = random.sample(population, tournament_size)
            parent1 = max(tournament1, key=lambda item: item.fitness)
            parent2 = max(tournament2, key=lambda item: item.fitness)

            if random.random() < self.config.crossover_rate:
                child = self.crossover(parent1, parent2)
            else:
                child = copy.deepcopy(random.choice([parent1, parent2]))
            child = self.mutate(child)
            self.calculate_fitness(child)
            new_population.append(child)
        return new_population

    def search(self) -> NeuralArchitecture:
        """Run the configured evolutionary search and return its best candidate."""
        self.best_accuracy_history = []
        self.avg_accuracy_history = []
        self.diversity_history = []
        self.best_architecture = None

        print("Starting evolutionary Neural Architecture Search")
        print(f"Population size: {self.config.population_size}")
        print(f"Generations: {self.config.generations}")
        print(f"Convolution filter choices: {self.conv_filters}")
        print(f"Input shape: {self.input_shape}")
        print(f"Number of classes: {self.num_classes}")

        population = self.initialize_population()
        for generation in range(self.config.generations):
            best_accuracy = max(item.accuracy for item in population)
            average_accuracy = sum(item.accuracy for item in population) / len(population)
            diversity = self.calculate_diversity(population)
            self.best_accuracy_history.append(best_accuracy)
            self.avg_accuracy_history.append(average_accuracy)
            self.diversity_history.append(diversity)

            generation_best = max(population, key=lambda item: item.fitness)
            if (
                self.best_architecture is None
                or generation_best.fitness > self.best_architecture.fitness
            ):
                self.best_architecture = copy.deepcopy(generation_best)

            print(
                f"Generation {generation + 1:2d}: "
                f"best proxy accuracy = {best_accuracy:.4f}, "
                f"mean proxy accuracy = {average_accuracy:.4f}, "
                f"diversity = {diversity:.3f}"
            )
            population = self.evolve_population(population)

        final_best = max(population, key=lambda item: item.fitness)
        if (
            self.best_architecture is None
            or final_best.fitness > self.best_architecture.fitness
        ):
            self.best_architecture = copy.deepcopy(final_best)

        print("\nSearch completed.")
        print(f"Best simulated accuracy: {self.best_architecture.accuracy:.4f}")
        print(f"Parameter count: {self.best_architecture.parameters:,}")
        print(f"Estimated inference time: {self.best_architecture.inference_time * 1000:.2f} ms")
        print(f"Fitness score: {self.best_architecture.fitness:.6f}")
        return self.best_architecture

In [ ]:
nas_config = NASConfig(
    population_size=20,
    generations=15,
    mutation_rate=0.3,
    crossover_rate=0.7,
    elite_size=4,
    max_layers=10,
    max_filters=512,
    max_dense_units=1024,
)

nas = NeuralArchitectureSearch(
    config=nas_config,
    input_shape=(224, 224, 3),
    num_classes=8,
)

start_time = time.time()
best_architecture = nas.search()
search_time = time.time() - start_time

print(f"\nElapsed search time: {search_time:.2f} seconds")
print("\n=== BEST CANDIDATE FOUND ===")
print(f"Number of layers: {len(best_architecture.layers)}")
print(f"Parameter count: {best_architecture.parameters:,}")
print(f"Simulated accuracy: {best_architecture.accuracy * 100:.2f}%")
print(f"Estimated inference time: {best_architecture.inference_time * 1000:.2f} ms")
print(f"Fitness score: {best_architecture.fitness:.6f}")

print("\nLayer sequence:")
for index, layer in enumerate(best_architecture.layers, start=1):
    if layer.layer_type == "conv":
        description = (
            f"Conv2D: {layer.filters} filters, "
            f"{layer.kernel_size}x{layer.kernel_size} kernel, "
            f"stride={layer.stride}, activation={layer.activation}"
        )
    elif layer.layer_type == "dense":
        description = f"Dense: {layer.units} units, activation={layer.activation}"
    elif layer.layer_type == "pool":
        description = f"Pooling: {layer.pool_size}x{layer.pool_size} window"
    elif layer.layer_type == "attention":
        description = "Simplified attention"
    else:
        description = f"Dropout: rate={layer.dropout_rate:.1f}"
    print(f"{index:2d}. {description}")

improvement = (best_architecture.accuracy - nas.baseline_accuracy) * 100
parameter_reduction = (
    (nas.baseline_params - best_architecture.parameters) / nas.baseline_params * 100
)
speed_improvement = (
    (0.01 - best_architecture.inference_time) / 0.01 * 100
)
print("\n=== COMPARISON WITH THE SIMULATION REFERENCE ===")
print(f"Proxy-accuracy difference: {improvement:+.2f} percentage points")
print(f"Parameter-count difference: {parameter_reduction:+.1f}%")
print(f"Estimated inference-time difference: {speed_improvement:+.1f}%")
print(f"Reference proxy accuracy: {nas.baseline_accuracy * 100:.1f}%")
print(f"Candidate proxy accuracy: {best_architecture.accuracy * 100:.1f}%")
print("These comparisons use simulation assumptions, not measured model performance.")

In [ ]:
def create_nas_visualizations(
    best_architecture: NeuralArchitecture,
    nas: NeuralArchitectureSearch,
    search_time: float,
):
    """Visualize evolutionary histories and simulated candidate metrics."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Evolutionary NAS: Simulated Search Results", fontsize=16)

    generations = range(1, len(nas.best_accuracy_history) + 1)
    ax = axes[0, 0]
    ax.plot(generations, nas.best_accuracy_history, marker="o", label="Best proxy accuracy")
    ax.plot(generations, nas.avg_accuracy_history, linestyle="--", label="Mean proxy accuracy")
    ax.axhline(nas.baseline_accuracy, color="green", linestyle=":", label="Reference proxy")
    ax.set_title("Proxy Accuracy by Generation")
    ax.set_xlabel("Generation")
    ax.set_ylabel("Simulated accuracy (fraction)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(generations, nas.diversity_history, color="purple", marker="o")
    ax.set_title("Population Structural Diversity")
    ax.set_xlabel("Generation")
    ax.set_ylabel("Mean structural-difference score")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    type_counts = defaultdict(int)
    for layer in best_architecture.layers:
        type_counts[layer.layer_type] += 1
    ax.pie(type_counts.values(), labels=type_counts.keys(), autopct="%1.1f%%", startangle=90)
    ax.set_title("Best Candidate: Layer-Type Composition")

    ax = axes[1, 1]
    metrics = ["Accuracy (%)", "Parameters (millions)", "Inference time (ms)"]
    candidate_values = [
        best_architecture.accuracy * 100,
        best_architecture.parameters / 1_000_000,
        best_architecture.inference_time * 1000,
    ]
    reference_values = [
        nas.baseline_accuracy * 100,
        nas.baseline_params / 1_000_000,
        10.0,
    ]
    x = np.arange(len(metrics))
    width = 0.35
    ax.bar(x - width / 2, candidate_values, width, label="Search candidate")
    ax.bar(x + width / 2, reference_values, width, label="Simulation reference")
    ax.set_title("Simulated Metrics (Different Units)")
    ax.set_ylabel("Metric value; units vary by category")
    ax.set_xticks(x, metrics)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(max(12, len(best_architecture.layers) * 1.5), 4))
    color_map = {
        "conv": "lightblue",
        "dense": "lightcoral",
        "pool": "lightgreen",
        "attention": "wheat",
        "dropout": "plum",
    }
    for index, layer in enumerate(best_architecture.layers):
        label = layer.layer_type
        if layer.layer_type == "conv":
            label += f"\n{layer.filters} filters\n{layer.kernel_size}x{layer.kernel_size}"
        elif layer.layer_type == "dense":
            label += f"\n{layer.units} units"
        elif layer.layer_type == "pool":
            label += f"\n{layer.pool_size}x{layer.pool_size}"
        elif layer.layer_type == "dropout":
            label += f"\nrate {layer.dropout_rate:.1f}"

        x_position = index * 1.4
        rectangle = plt.Rectangle(
            (x_position, 0),
            1.0,
            1.0,
            facecolor=color_map[layer.layer_type],
            edgecolor="black",
        )
        ax.add_patch(rectangle)
        ax.text(x_position + 0.5, 0.5, label, ha="center", va="center", fontsize=9)
        if index < len(best_architecture.layers) - 1:
            ax.annotate(
                "",
                xy=(x_position + 1.35, 0.5),
                xytext=(x_position + 1.02, 0.5),
                arrowprops={"arrowstyle": "->"},
            )
    ax.set_xlim(-0.2, len(best_architecture.layers) * 1.4)
    ax.set_ylim(-0.2, 1.2)
    ax.set_title("Sequential Candidate Architecture (Schematic)")
    ax.set_xlabel("Layer order; block colors indicate layer type")
    ax.set_yticks([])
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Candidate Analysis: Simulation-Based Estimates", fontsize=15)

    differences = [
        (accuracy - nas.baseline_accuracy) * 100
        for accuracy in nas.best_accuracy_history
    ]
    axes[0].plot(generations, differences, marker="o")
    axes[0].axhline(0, color="red", linestyle="--")
    axes[0].set_title("Proxy Accuracy Difference from Reference")
    axes[0].set_xlabel("Generation")
    axes[0].set_ylabel("Difference (percentage points)")
    axes[0].grid(True, alpha=0.3)

    parameter_values = [
        best_architecture.parameters / 1_000_000,
        nas.baseline_params / 1_000_000,
    ]
    bars = axes[1].bar(
        ["Search candidate", "Simulation reference"],
        parameter_values,
        color=["blue", "red"],
    )
    axes[1].set_title("Parameter-Count Estimate")
    axes[1].set_ylabel("Parameters (millions)")
    axes[1].tick_params(axis="x", rotation=15)
    axes[1].grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, parameter_values):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.2f}M", ha="center", va="bottom")

    evaluations = (
        nas_config.population_size
        + nas_config.generations
        * (nas_config.population_size - nas_config.elite_size)
    )
    labels = ["Evaluations", "Elapsed time (s)", "Fitness"]
    values = [evaluations, search_time, best_architecture.fitness]
    normalized = [evaluations / 1000, search_time / 10, best_architecture.fitness * 100]
    bars = axes[2].bar(labels, normalized, color=["orange", "purple", "green"])
    axes[2].set_title("Search-Budget Indicators (Normalized)")
    axes[2].set_ylabel("Display-scaled value; scales differ")
    axes[2].tick_params(axis="x", rotation=15)
    axes[2].grid(True, axis="y", alpha=0.3)
    actual_labels = [f"{evaluations:,}", f"{search_time:.2f} s", f"{best_architecture.fitness:.4f}"]
    for bar, label in zip(bars, actual_labels):
        axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha="center", va="bottom")

    fig.tight_layout()
    plt.show()


create_nas_visualizations(best_architecture, nas, search_time)

In [ ]:
def analyze_architecture_complexity(
    architecture: NeuralArchitecture,
) -> dict:
    """Summarize layer composition and simplified computational estimates."""
    layer_analysis = defaultdict(int)
    total_filters = 0
    total_dense_units = 0
    attention_count = 0

    for layer in architecture.layers:
        layer_analysis[layer.layer_type] += 1
        if layer.layer_type == "conv":
            total_filters += layer.filters
        elif layer.layer_type == "dense":
            total_dense_units += layer.units
        elif layer.layer_type == "attention":
            attention_count += 1

    def calculate_flops(candidate: NeuralArchitecture) -> int:
        """Estimate multiply-add operations for convolution and dense layers."""
        total_flops = 0
        current_shape = candidate.input_shape
        for layer in candidate.layers:
            if layer.layer_type == "conv":
                height, width, input_channels = current_shape
                output_h = (height - layer.kernel_size) // layer.stride + 1
                output_w = (width - layer.kernel_size) // layer.stride + 1
                total_flops += (
                    output_h
                    * output_w
                    * layer.filters
                    * layer.kernel_size**2
                    * input_channels
                )
                current_shape = (output_h, output_w, layer.filters)
            elif layer.layer_type == "pool":
                height, width, channels = current_shape
                current_shape = (
                    height // layer.pool_size,
                    width // layer.pool_size,
                    channels,
                )
            elif layer.layer_type == "dense":
                input_size = int(np.prod(current_shape))
                total_flops += input_size * layer.units * 2
                current_shape = (layer.units,)
        return total_flops

    total_flops = calculate_flops(architecture)
    parameters = architecture.calculate_parameters()
    memory_mb = parameters * 4 / (1024**2)
    estimated_throughput = 10_000_000_000 / total_flops if total_flops else 0.0

    print("=== ARCHITECTURE COMPLEXITY ANALYSIS ===")
    print("Layer distribution:")
    for layer_type, count in sorted(layer_analysis.items()):
        print(f"  {layer_type.capitalize()}: {count} layers ({count / len(architecture.layers) * 100:.1f}%)")
    print(f"Total convolutional filters: {total_filters:,}")
    print(f"Total dense units, including the classifier: {total_dense_units:,}")
    print(f"Attention layers: {attention_count}")
    print(f"Parameters per layer, on average: {parameters / len(architecture.layers):,.0f}")
    print(f"Simplified operation estimate: {total_flops:,} multiply-add-related operations")
    print(f"Operations per parameter: {total_flops / parameters:.2f}" if parameters else "Operations per parameter: undefined (zero parameters)")
    print(f"Parameter storage estimate at 4 bytes each: {memory_mb:.2f} MiB")
    print(f"Illustrative throughput estimate: {estimated_throughput:.1f} images/second")
    print("Throughput is a stated rule-of-thumb estimate, not a hardware benchmark.")

    return {
        "layer_distribution": dict(layer_analysis),
        "total_filters": total_filters,
        "total_dense_units": total_dense_units,
        "attention_count": attention_count,
        "total_flops": total_flops,
        "memory_mb": memory_mb,
        "estimated_throughput": estimated_throughput,
    }


complexity_analysis = analyze_architecture_complexity(best_architecture)

In [ ]:
def summarize_evolutionary_search(
    architecture: NeuralArchitecture,
    nas: NeuralArchitectureSearch,
    config: NASConfig,
    elapsed_time: float,
) -> pd.DataFrame:
    """Report search-budget and candidate metrics for the evolutionary run."""
    evaluations = config.population_size + config.generations * (
        config.population_size - config.elite_size
    )
    results = pd.DataFrame(
        [
            {
                "Measure": "Fitness evaluations",
                "Value": f"{evaluations:,}",
                "Interpretation": "Initial population plus evaluated offspring",
            },
            {
                "Measure": "Elapsed time (seconds)",
                "Value": f"{elapsed_time:.2f}",
                "Interpretation": "Runtime of this simulated search",
            },
            {
                "Measure": "Best simulated accuracy",
                "Value": f"{architecture.accuracy * 100:.2f}%",
                "Interpretation": "Proxy value, not validation accuracy",
            },
            {
                "Measure": "Best-candidate parameters",
                "Value": f"{architecture.parameters:,}",
                "Interpretation": "Count under the stated layer assumptions",
            },
            {
                "Measure": "Best-candidate fitness",
                "Value": f"{architecture.fitness:.6f}",
                "Interpretation": "Weighted proxy objective used for selection",
            },
            {
                "Measure": "Recorded generations",
                "Value": str(len(nas.best_accuracy_history)),
                "Interpretation": "Populations recorded before each evolution step",
            },
        ]
    )
    print("=== EVOLUTIONARY SEARCH SUMMARY ===")
    print(results.to_string(index=False))
    return results


search_summary = summarize_evolutionary_search(
    best_architecture,
    nas,
    nas_config,
    search_time,
)

## Strengths, limitations, and appropriate use

### What the method demonstrates

Evolutionary NAS automates the exploration of a discrete architecture space. In this example, architecture sequences are the individuals; simulated fitness provides a ranking signal; tournament selection favors stronger candidates without requiring a gradient through the architecture choices; crossover recombines layer sequences; mutation introduces structural variation; and elitism carries high-fitness candidates forward. The recorded accuracy and diversity histories help describe how the population changes during the search.

### Limitations of this demonstration

- The accuracy proxy is a hand-designed formula. It does not represent learned visual features, validation performance, or threat-detection reliability.
- The search does not instantiate or train neural networks. Its parameter, operation, memory, and inference-time figures are estimates under simplified assumptions.
- The layer model is sequential and omits details such as normalization, residual connections, padding choices beyond the stated valid-convolution rule, and implementation-specific attention computations.
- Random variation affects simulated fitness, and the run is not seeded. A candidate's fitness therefore reflects both architecture features and simulation noise.
- A high score in this proxy objective does not establish safety, robustness, calibration, interpretability, or suitability for operational security screening.

### When the general approach may be relevant

Architecture search can be considered when a genuine learning task has suitable labeled data, a carefully defined architecture space, and a reliable evaluation procedure. For security applications, evaluation would also need to account for class imbalance, missed-threat costs, false alarms, distribution shift, and independent testing. Those elements are outside the scope of this simulated notebook.

### Summary

This notebook presents an evolutionary NAS workflow and the role of its principal steps: candidate generation, fitness evaluation, tournament selection, crossover, mutation, elitism, and result analysis. It provides a runnable demonstration of the search mechanics, while clearly distinguishing simulated metrics from empirical model performance. No detection-accuracy claim can be inferred from the proxy results.